# AOP-ProCon — Step 1: Dataset Curation → AOP-BenchPos (S1) and AOP-BenchPos_S2   *(validated build v3)*
**Component 1 — Fernando W A A T — IT23144408 — J26-DS-302 (PlantAOx-RAISE)**

> **Status.** The previous build (v2) was run end-to-end on a fresh Colab session and its S2 matched the offline replay exactly (canonical fingerprint `a1cf5030e72ef73b`).
> **This build (v3) changes three things** — the near-duplicate rule now also links 3-aa ↔ 4-aa pairs, `FLQL` is removed only from the mechanism-labelled tiers, and negatives are flagged against the *raw* database pool.
> Its expected fingerprint (`47d9c2a2fff63eb1`) comes from the offline replay and **still has to be confirmed by one Colab run**. Until then, call it "validated by replay", not "final".

This notebook builds the curated, mechanism-tagged, de-duplicated antioxidant-peptide benchmark used by every later
step. It is organised to match `progress_log.md`:

`1.1` source collection → `1.2` merge → `1.3` de-duplication → `1.4` sufficiency → `1.5` AOP-BenchPos (S1) →
`1.6` leak-safe homology grouping and 2-fold split → `1.7` S2 multi-label dataset → `1.8` negatives → `1.9` manifest.

## Outputs (all written to `BASE = /content/drive/MyDrive/AOP-ProCon` → `data/processed/`)

| File | What it is |
|---|---|
| `AOP-BenchPos.csv` | **S1**: one row per (sequence, tier). Intermediate / reference only. |
| `AOP-BenchPos_S2.csv` | **S2**: one row per unique sequence, multi-label `labels` ∈ {FRS, MC, FRS;MC, GEN}. **The dataset every later step uses.** |
| `AOP-Inactive.csv` | AnOxPePred's experimentally validated *non-antioxidant* peptides, with leak-safe folds. **Only the `db_conflict == False` ones are clean negatives**: `db_conflict` is checked against the *raw* database pools, and most of the peptides are listed as antioxidant by a Tier-3 database (§1.5b). |
| `AOP-Conflicts.csv` | Tier-3 sequences that AnOxPePred reports as experimentally inactive (V20), with the databases that list them. |
| `AOP-Provenance.csv` | **Every** (sequence, tier, source, source reference) before de-duplication, so complete source membership is never lost. |
| `excluded_by_primary_evidence.csv` | Rows removed from the mechanism-labelled tiers (no detectable FRS/MC activity in the cited assays) or for non-canonical residues, with the reason. |
| `AOP-BenchPos.previous.csv`, `AOP-BenchPos_S2.previous.csv`, `s2_change_log.csv` | Backups of the previous S1 / S2 (taken once, never overwritten) and a row-level diff of S2, so downstream artefacts can be re-aligned. |
| `step1_manifest.json` | Counts, SHA-256 of every output, S2 fingerprint, git commits of the cloned sources, run date. |

## What was validated in this version (every item re-checked against the primary source)

| ID | Finding | Action in this notebook |
|---|---|---|
| **V1** | `FLQL` (Sonklin et al. 2021, Table 1): **ND (no detectable activity) in DRSA, HRSA, SRSA and metal-chelating activity**; FRAP 0.01 (reducing power only). It was in Tier 2 *and* labelled dual FRS+MC. | Removed from the **mechanism-labelled tiers (Tier 1/2)** (§1.1.6, §1.2) and logged in `excluded_by_primary_evidence.csv`. Wording used throughout: *"no detectable FRS or MC activity in the cited assays; excluded from the mechanism-labelled benchmark"* — **not** a claim that the peptide has no antioxidant activity. A database-level Tier-3 listing, if any, is left unchanged. |
| **V2** | The 21 mung-bean peptides were all blanket-reclassified as dual. Per-peptide data show **20 of 21 have detectable chelation *and* radical-scavenging activity**; one (`FLQL`) has neither. | Labels are now **derived from the transcribed per-peptide tables** by an explicit rule (§1.1.5–1.1.6), not assumed. |
| **V3** | AnOxPePred (Olsen et al. 2020) labels a peptide FRS / Chelator = 1 *"if their source had measured/indicated an activity"* — it is **not DPPH/ABTS-specific**. | `evidence_tier` for AnOxPePred rows is now `per_peptide_source_reported_activity` (the old `..._dpph_abts_...` / `..._edta_type_...` strings over-claimed). |
| **V4** | Citation error: the sunflower-meal paper's first author is **Camaño Echavarría** (8 authors; none named Enríquez). | Corrected everywhere, including the `source_db` value. |
| **V5** | Walnut (Fan et al. 2023): the 27 peptides were **identified by LC-MS/MS in the Fe-chelate fraction and docked in silico** — not individually tested. | Kept as *candidate* chelators (weight 0.5, `mc_evidence = candidate`). |
| **V6** | Rice bran (Fernandez et al. 2025): all 9 peptides were **synthesised and individually tested** for Cu(II) binding (all ≥ 10 % chelated). | `mc_evidence = validated`. |
| **V7** | 40 %-identity CD-HIT is not used below 10 aa, so every short peptide was its own cluster; near-duplicates could sit on both sides of the split. The old note calling this "conservative" was wrong *for leakage*. | New **homology groups**: CD-HIT clusters ∪ near-duplicate pairs. Near-duplicate = **normalised edit similarity (1 − Levenshtein ÷ longer length) ≥ 70 %** with at least one of the two peptides ≥ 4 aa (so a 3-aa peptide is linked to a 4-aa peptide that differs by one residue). *The 70 % number is borrowed from AnOxPePred, but their identity is a different quantity (Needleman–Wunsch: identical = 1, dissimilar = 0, gap −10, end gaps free). This is **our own rule — not the same method, and not the proposal's 40 % identity separation.*** Folds are assigned per group and re-verified independently (§1.6.2). |
| **V8** | 218 AnOxPePred peptides labelled FRS = 0 **and** Chelator = 0 are *"experimentally-validated non-antioxidant"* peptides (paper's wording). They were discarded. | Kept as `AOP-Inactive.csv` with leak-safe folds (§1.8). |
| **V9** | `BASE_RAW` was used but **never defined** — a fresh *Restart & Run all* raised `NameError` in the setup cell. | The variable is gone: one `BASE`, every path derived from it in one cell. |
| **V10** | "Tier 2 target (≥ 150) met exactly" — reached partly by a mislabelled peptide. | Count is reported as-is. **No peptide is added or relabelled to reach a target.** |
| **V11** | Hard-coded numbers in markdown went stale (e.g. DFBP-out "train=485"). | Numbers now come from printed output only. |
| **V12** | Run-to-run provenance was implicit. | Git commit hashes, backup of the previous S2, change log, manifest and fingerprint. |
| **V13** | The notebook said CD-HIT reporting 100 % identity between peptides of different lengths is "impossible". It is not: CD-HIT's default identity is *identical residues ÷ length of the **shorter** sequence* (CD-HIT user guide, `-c` / `-G`), so a peptide contained in a longer one scores 100 %. | Explanation corrected (§1.3.3). The real reasons CD-HIT is avoided below 10 aa are its minimum-length handling (it did not process the 3-mers) and the shorter-sequence denominator. |
| **V14** | `drop_duplicates(["sequence","tier"])` keeps the first source only, and AnOxPePred's own `Source` (reference) column was dropped. In the raw Tier-3 pool, 1,359 of 1,696 sequences occur in more than one database, so a per-database breakdown of the *retained* source is misleading (e.g. DFBP shows 36 retained sequences but lists 484). | New **provenance table** (`AOP-Provenance.csv`: one row per sequence × tier × source, with the source reference) and S2 columns `all_sources`, `n_sources` (§1.2, §1.7). |
| **V15** | `sample_weight = 0.5` is a modelling choice, not evidence validation. | S2 flag `strict_evidence` separates individually supported labels (**primary** analysis) from candidates (**sensitivity** analysis) (§1.7). |
| **V16** | Tripeptides are about 36 % of the mechanism-active set. They **can** be grouped (one substitution = 67 % identity), but the links join 229 of the 259 tripeptides into one giant component, so no balanced split exists. AnOxPePred's authors observed the same: below ~66 % identity *"the high number of 3′mers … all ends up in one cluster prohibiting the creation of 5 even partitions"*. | Reported as a **feasibility limitation, not an impossibility**. 3-aa ↔ 3-aa pairs are *not* linked (67 % < 70 %); 3-aa ↔ 4-aa pairs are. S2 flag `homology_resolvable` (≥ 4 aa) supports a **strict length-subset experiment**: exclude 3-aa peptides from training, validation **and** the retrieval gallery, then retrain (§1.6.2, §1.6.2b). |
| **V17** | The Tier-1/2 split is a *homology-grouped, MC-stratified 2-fold split*, **not** leave-source-out. | The column name `lso_fold` is kept only for backward compatibility with Steps 2–5; the manifest records `split_type`. Say "grouped 2-fold split" in the write-up. |
| **V18** | The PMC table fetch silently fell back to "the table with most sequences" when the id was not found; Peptipedia was read from a different raw folder than every other source; non-canonical residues were checked only for the scraped sources. | Exact table id **and** exact expected count are mandatory; Peptipedia is searched in both folders and its path printed; one global canonical-residue check covers every source (§1.2). |
| **V19** | Per-tier CD-HIT could in principle drop a sequence from one tier while keeping it in the other, silently losing a dual label. | A **label-loss audit** (§1.5) counts and asserts it; on the validated data it found 41 sequences removed by de-duplication (all ≥ 10 aa) and none retained under another tier. |
| **V20** | **AnOxPePred's experimentally-validated non-antioxidant peptides are also listed as "antioxidant" by Tier-3 databases.** In the raw database pools 190 of the 217 are listed (189 DFBP / 179 Multi-AOP; 146 of the 218 come from a single source paper); after the length filter and de-duplication, **184 sequences — 21.4 % of Tier 3 — contradict a per-peptide negative.** | New conflict audit (§1.5b). Default policy: the retained conflicting sequences are **removed from the Tier-3 (GEN) pool** and written to `AOP-Conflicts.csv` (`CONFLICT_POLICY = "keep"` retains and flags them). **Negatives are flagged against the *raw* database pool (before de-duplication)**, so a peptide a database lists but de-duplication happened to drop is not mislabelled "clean". |

## Known limitations (state these yourself in the viva)
1. **Tripeptides (about 36 % of the mechanism-active set) cannot be split into balanced, homology-safe folds** (V16): they can be grouped, but form one giant component. Repeat every downstream result on `homology_resolvable == True` **with the 3-aa peptides excluded from training, validation and the retrieval gallery**.
2. **One source dominates.** Almost all FRS-only peptides and about half of the MC-active peptides come from AnOxPePred; the rest of MC-active come from five small literature sources with different peptide lengths. Any FRS-vs-MC contrast is therefore partly a source / length contrast — control for `source_group` and `length`.
3. **45 MC labels are candidates** (sunflower, walnut: identified in an active fraction or chelate, not tested alone). Use `strict_evidence` for the primary analysis (V15).
4. **Tier 3 has no mechanism ground truth, and its "antioxidant" label is database-level.** About a fifth of the original Tier 3 contradicted AnOxPePred's own negatives (V20); further undetected conflicts are possible. Analyses on it are exploratory similarity analyses, not accuracy measurements.
5. **A matching fingerprint (§1.9) proves reproducibility only** — not label correctness, absence of leakage, or model validity.
7. **The homology rule is our own** normalised-edit-similarity rule (V7): it is neither the proposal's 40 % identity separation nor AnOxPePred's alignment identity. State it as such, and state exactly what is audited in §1.6.2b.
6. **The sunflower list (29 sequences) is hand-transcribed** from the paper's Table 3; the publisher's repository blocks automated download, so it could not be machine-verified. Re-check it against the paper before submission.

## Setup
One cell defines every path from a single variable, `BASE = /content/drive/MyDrive/AOP-ProCon`. Raw third-party downloads go to `BASE/data/raw`, every derived output to `BASE/data/processed`. (There is no separate `V2` or raw-data folder.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, sys, json, hashlib, subprocess, random, time, datetime
import numpy as np, pandas as pd, requests
from io import StringIO

# --- folder layout (V9): ONE base path; every other path is derived from it ---------------------------------------
BASE    = "/content/drive/MyDrive/AOP-ProCon"
RAW_DIR = f"{BASE}/data/raw"            # raw third-party downloads (git clones, scraped pages, the manual Peptipedia export)
PROC    = f"{BASE}/data/processed"      # every derived output
for d in (RAW_DIR, PROC, f"{BASE}/data/embeddings", f"{BASE}/results"):
    os.makedirs(d, exist_ok=True)

RUN_DATE = datetime.date.today().isoformat()
PROVENANCE = {"run_date": RUN_DATE, "notebook": "STEP01_V2_Dataset_Curation_FINAL"}
pd.set_option("display.max_colwidth", 80)
def find_raw(filename):
    """Locate a manually supplied input in the raw folder and say where it is (V18)."""
    path = f"{RAW_DIR}/{filename}"
    if not os.path.exists(path):
        raise FileNotFoundError(f"{filename} not found in {RAW_DIR}")
    print(f"found {filename} in {RAW_DIR}"); return path

print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "| run", RUN_DATE)
print("BASE:", BASE)

In [ ]:
# --- evidence vocabulary (V3) --------------------------------------------------------------------------
EV_SOURCE  = "per_peptide_source_reported_activity"   # AnOxPePred: label = 1 if the source study measured/indicated activity
EV_PEPTIDE = "per_peptide_functional_assay"           # synthesised and tested individually in the source paper
EV_COMPLEX = "complex_or_hydrolysate_level_assay"     # peptide identified in an active fraction / chelate; not tested alone
EV_DB      = "database_binary_label"                  # Tier 3: a database says "antioxidant"; mechanism unknown

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")
def is_valid_seq(s):
    s = str(s).strip().upper()
    return 2 <= len(s) <= 60 and s != "" and all(c in VALID_AA for c in s)

def git_clone(url, name):
    """Clone once, record the commit hash (reproducibility)."""
    dest = f"{RAW_DIR}/{name}"
    if os.path.isdir(dest):
        print("already cloned:", dest)
    else:
        subprocess.run(["git", "clone", "--depth", "1", url, dest], check=True)
    commit = subprocess.run(["git", "-C", dest, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    PROVENANCE[f"git_{name}"] = commit
    print(f"{name}: commit {commit[:12]}")
    return dest
# Optional: an earlier S2 file to diff against (e.g. the old 1,573-row file). If None, an S2 that already exists in PROC is backed up and used.
PREVIOUS_S2_PATH = None
print("constants ready")

---
## 1.1 Data Source Collection
Five primary sources (AnOxPePred, Multi-AOP, DFBP, Peptipedia, BIOPEP-UWM) plus **five supplementary Tier-2 literature
sources** added after the Tier-2 shortfall was identified (§1.4). The supplementary sources were each re-read from the
primary paper in this validation pass (see the table in the header).

**Tiers.** Tier 1 = free-radical scavenging (FRS); Tier 2 = metal chelation (MC); Tier 3 = confirmed antioxidant, mechanism
*unknown* (query-only, label `GEN` in S2).

### 1.1.1 AnOxPePred
The Dataset Guide's link (`dtu-pas/AnOxPePred`) is dead; the repository lives under the first author's account. The paper
(Olsen et al. 2020, *Sci Rep* 10:21471, doi:10.1038/s41598-020-78319-w) defines the labels: a class is **1 "if their source had
measured/indicated an activity" and 0 otherwise** (V3). It also reports 218 *experimentally-validated non-antioxidant* peptides.

In [ ]:
anox_dir = git_clone("https://github.com/TobiasHeOl/AnOxPePred.git", "AnOxPePred")

# Use the clean, pre-deduplicated file only -- NOT the raw/intermediate files that sit next to it
df = pd.read_csv(f"{anox_dir}/Data/Supplementary_Data_S1_Marcatili.csv")
print(df.shape, df.columns.tolist())      # expected (917, 5): ['Unnamed: 0', 'Sequence', 'Source', 'FRS', 'Chelator']

In [ ]:
df["Sequence"] = df["Sequence"].astype(str).str.strip().str.upper()
bad = ~df["Sequence"].apply(lambda s: s != "" and set(s) <= VALID_AA)
print("rows with non-standard residues (dropped):", int(bad.sum()))
df = df[~bad].reset_index(drop=True)

is_inactive = (df["FRS"] == 0) & (df["Chelator"] == 0)
is_dual     = (df["FRS"] == 1) & (df["Chelator"] == 1)
print("FRS = 0 and Chelator = 0 (experimentally validated NON-antioxidant -> kept as negatives, Section 1.8):", int(is_inactive.sum()))
print("FRS = 1 and Chelator = 1 (dual mechanism, kept in BOTH tiers):", int(is_dual.sum()))

# Tier 1 = source-reported free-radical scavenging; Tier 2 = source-reported metal chelation (V3)
# V14: AnOxPePred's own `Source` column (the reference each peptide came from) is kept as `source_ref`
df["source_ref"] = df["Source"].astype(str)
tier1_from_anoxpepred = df[df["FRS"] == 1][["Sequence", "source_ref"]].assign(tier="Tier1", evidence_tier=EV_SOURCE)
tier2_from_anoxpepred = df[df["Chelator"] == 1][["Sequence", "source_ref"]].assign(tier="Tier2", evidence_tier=EV_SOURCE)
anox_inactive = df[is_inactive][["Sequence"]].rename(columns={"Sequence": "sequence"}).drop_duplicates().reset_index(drop=True)
print("Tier 1 candidates:", len(tier1_from_anoxpepred), "| Tier 2 candidates:", len(tier2_from_anoxpepred),
      "| inactive controls:", len(anox_inactive))

### 1.1.2 Supplementary Tier-2 sources — how they are collected
Four of the five (walnut, rice bran, mung bean, mung-bean meal) are fetched programmatically: NCBI's `efetch` returns the
article's full-text XML (PubMed Central), and the table with the peptide sequences is parsed into a DataFrame. The column that
holds the sequences is auto-detected because header names differ between papers.

The fifth (sunflower meal) is hand-transcribed: the paper has no PMC deposit and its table is not machine-readable.

**Evidence strength is recorded per row** (`evidence_tier`), because the sources are *not* equally rigorous:

| `evidence_tier` | Meaning | Sources |
|---|---|---|
| `per_peptide_source_reported_activity` | Source study measured/indicated the activity for that peptide (AnOxPePred's own definition). | AnOxPePred (Tier 1 and 2) |
| `per_peptide_functional_assay` | Peptide synthesised and tested **individually** (chelation and/or radical assays). | Rice bran, mung bean, mung-bean meal |
| `complex_or_hydrolysate_level_assay` | Peptide **identified** in an active fraction or metal-chelate; not tested alone. Weakest evidence → `sample_weight = 0.5`. | Sunflower, walnut |
| `database_binary_label` | A database labels it antioxidant; mechanism unknown. | Multi-AOP, DFBP, Peptipedia |

In [ ]:
import re as _re, time
from io import StringIO

_AA = set("ACDEFGHIKLMNPQRSTVWY")

def is_valid_seq(s):
    s = str(s).strip().upper()
    return 2 <= len(s) <= 60 and s != "" and all(c in _AA for c in s)

def _efetch_pmc_xml(pmcid, retries=4):
    """Fetch a PMC article's full-text XML, politely rate-limited with retry/backoff.
    NCBI throttles anonymous callers to ~3 requests/sec and returns a short error body
    (not the article) when exceeded -- the usual cause of a 'table-wrap not found' error
    when several PMC fetches run back-to-back. We pace the calls, then verify we actually
    got an <article> XML before trusting it, retrying with exponential backoff if not.
    (Optionally put your email in `email=` below -- NCBI asks for it but it is not required.)"""
    url = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
           f"?db=pmc&id={pmcid}&rettype=full&retmode=xml&tool=AOP-ProCon&email=")
    last = ""
    for attempt in range(retries):
        time.sleep(0.5)                                   # stay under NCBI's 3 req/sec
        try:
            xml = requests.get(url, timeout=40, headers={"User-Agent": "Mozilla/5.0"}).text
        except Exception as e:
            last = str(e); time.sleep(2 ** attempt); continue
        if "<article" in xml and "<table-wrap" in xml:    # a real article, not a throttle error
            return xml
        last = xml[:200]
        time.sleep(2 ** attempt)                          # back off, then retry
    raise RuntimeError(f"PMC{pmcid}: no valid article XML after {retries} tries "
                       f"(likely NCBI rate-limit). Last response head: {last!r}")

def _parse_table_sequences(table_html):
    """Parse one HTML table and return the amino-acid sequences from whichever column
    holds the most valid peptide strings (header names differ paper to paper)."""
    try:
        df = pd.read_html(StringIO(table_html))[0]
    except Exception:
        return []
    best_col, best = None, 0
    for col in df.columns:
        n = df[col].apply(is_valid_seq).sum()
        if n > best:
            best_col, best = col, n
    if best_col is None:
        return []
    return sorted(set(df[best_col][df[best_col].apply(is_valid_seq)].str.upper()))

def fetch_pmc_table_sequences(pmcid, table_wrap_id, expected_n):
    # Return the peptide sequences of ONE named table of a PMC article. STRICT (V18): the table id must exist and the
    # number of sequences must equal expected_n; there is no fallback to 'whichever table has most sequences'.
    xml = _efetch_pmc_xml(pmcid)
    blocks = []
    for m in _re.finditer(r"<table-wrap\b[^>]*>", xml):
        wid_m = _re.search(r'\bid="([^"]+)"', m.group(0))
        wid = wid_m.group(1) if wid_m else None
        end = xml.find("</table-wrap>", m.start()) + len("</table-wrap>")
        tw = xml[m.start():end]
        t0, t1 = tw.find("<table"), tw.find("</table>")
        if t0 != -1 and t1 != -1:
            blocks.append((wid, tw[t0:t1 + 8]))
    for wid, thtml in blocks:
        if wid == table_wrap_id:
            seqs = _parse_table_sequences(thtml)
            assert len(seqs) == expected_n, (f"PMC{pmcid} table {wid}: parsed {len(seqs)} sequences, expected {expected_n} "
                                             f"-- the article or the parser changed; inspect manually")
            print(f"  PMC{pmcid} (table {wid}): {len(seqs)} sequences")
            return seqs
    raise ValueError(f"PMC{pmcid}: table {table_wrap_id!r} not found. Present ids: {[w for w, _ in blocks]}")

### 1.1.2a Sunflower-meal Cu²⁺-chelating peptides
**Source (corrected, V4):** Camaño Echavarría, Mathé, Arnoux, Stefan, Paris, Selmeczi, Udenigwe, Canabady-Rochelle (2025).
*Metal-Chelating Peptides Derived from Sunflower Meal Protein: Preparation, Isolation, Identification, and Antioxidant
Properties.* **ACS Food Science & Technology 5(6):2336–2350.** doi:10.1021/acsfoodscitech.5c00191 (citation checked against Crossref).
(An earlier version of this notebook cited the first author as "Enríquez"; that name does not appear among the authors.)

His-containing peptides were identified by IMAC-Cu²⁺ affinity chromatography + LC-MS/MS from hydrolysate fractions; the
paper frames Cu²⁺ chelation as an antioxidant mechanism (suppressing Fenton/Haber-Weiss chemistry) and measures antioxidant
activity on the *hydrolysate*, not on each peptide. Evidence tier: **complex/hydrolysate-level** (candidate chelators).

**Open item:** transcribed by hand from Table 3. The publisher/repository blocks automated download, so this list could not be
re-verified by machine in the validation pass. The cell prints the list for a final manual tick-off.

In [ ]:
# Camano Echavarria et al. 2025, ACS Food Sci. Technol. 5(6):2336-2350, doi:10.1021/acsfoodscitech.5c00191, Table 3
sunflower_mcp_sequences = [
    # Fraction 2 (F2)
    "AGGISSEHIQGQQQ", "AIQSPH", "AVLAPH", "DFIRH", "FPILEH", "FPILEHL",
    "FSPH", "FVTPEEEQQQMH", "GENDQRGHIIF", "GENDQRGHIIFVQ", "GISSEHIQGQ",
    "GIVSPH", "GVDFIRH", "HNDGNTELVVVVF", "HVDQETASKL", "NIDNPSHADFVNPQ",
    "PHYA", "RGENDQRGHIIF", "SEVFH", "SEVFHVD", "SFLAGGISSEHIQGQQQ",
    "SPHWTIN", "WVHNDGNAE", "YANQLDPNHRRF",
    # Fraction 3 (F3)
    "ALFSPH", "DFIRHR", "GTAHWL", "GVDFIRHR", "NPSHADFV",
]
assert len(sunflower_mcp_sequences) == 29 and len(set(sunflower_mcp_sequences)) == 29
assert all(set(s) <= VALID_AA for s in sunflower_mcp_sequences)

sunflower_mcp_df = pd.DataFrame({"sequence": sunflower_mcp_sequences})
sunflower_mcp_df["tier"] = "Tier2"
sunflower_mcp_df["source_db"] = "Sunflower_MCP_CamanoEchavarria2025"
sunflower_mcp_df["evidence_tier"] = EV_COMPLEX
sunflower_mcp_df["source_ref"] = "doi:10.1021/acsfoodscitech.5c00191 (Table 3)"

print("Sunflower candidates loaded:", len(sunflower_mcp_df), "(expected 29)")
print("MANUAL CHECK before submission: tick these 29 against Table 3 of the paper ->")
print(", ".join(sunflower_mcp_sequences))

### 1.1.3 Walnut iron-chelating peptides (Fe²⁺)
**Source:** Fan et al. (2023), *Foods* 12(1):226, doi:10.3390/foods12010226 (PMC9818316). Table 2 lists 27 peptides **identified by
LC-MS/MS in the walnut peptide–Fe²⁺ chelate (WP-Fe)**; Table 3 is an **in-silico docking** table for the same 27.
The paper shows that the *chelate* scavenges DPPH and hydroxyl radicals better than the unchelated hydrolysate (a complex-level
result). The 27 peptides themselves were **not individually tested** (V5) → candidate chelators, evidence `complex_or_hydrolysate_level`,
`sample_weight = 0.5`.

In [ ]:
# PMC9818316, Table 2 (XML id foods-12-00226-t002)
walnut_mcp_sequences = fetch_pmc_table_sequences("9818316", "foods-12-00226-t002", expected_n=27)
walnut_mcp_df = pd.DataFrame({"sequence": walnut_mcp_sequences})
walnut_mcp_df["tier"] = "Tier2"
walnut_mcp_df["source_db"] = "Walnut_MCP_Fan2023"
walnut_mcp_df["evidence_tier"] = EV_COMPLEX
walnut_mcp_df["source_ref"] = "doi:10.3390/foods12010226 (Table 2)"
print("Walnut candidates loaded:", len(walnut_mcp_df), "(expected 27)")

### 1.1.4 Rice-bran copper-binding peptides (Cu²⁺)
**Source:** Fernandez et al. (2025), *RSC Advances* 15(37):30697–30710, doi:10.1039/d5ra05166e (PMC12394934). Table 1: nine peptides selected
from a Cu(II)-IMAC-enriched hydrolysate and **synthesised by solid-phase peptide synthesis**; each was **individually tested** for Cu(II)
binding (zincon competition assay) — *all nine chelated ≥ 10 % of the copper* (V6). A 3-CCA → 7-OH-CCA fluorescence assay of Cu-generated
hydroxyl radicals was run on the set. Evidence tier: per-peptide functional assay. Mechanism label: **MC only** (the radical assay is a
copper-ROS proxy, not a radical-scavenging measurement of the peptide, so FRS is *not* added).

In [ ]:
# PMC12394934, Table 1 (XML id tab1)
rice_bran_mcp_sequences = fetch_pmc_table_sequences("12394934", "tab1", expected_n=9)
rice_bran_mcp_df = pd.DataFrame({"sequence": rice_bran_mcp_sequences})
rice_bran_mcp_df["tier"] = "Tier2"
rice_bran_mcp_df["source_db"] = "RiceBran_MCP_Fernandez2025"
rice_bran_mcp_df["evidence_tier"] = EV_PEPTIDE
rice_bran_mcp_df["source_ref"] = "doi:10.1039/d5ra05166e (Table 1)"
print("Rice bran peptides loaded:", len(rice_bran_mcp_df), "(expected 9)")

### 1.1.5 Mung-bean iron-binding antioxidant peptides
**Source:** Chunkao et al. (2020), *Foods* 9(10):1406, doi:10.3390/foods9101406 (PMC7600763). Table 2: ten synthesised peptides, each with a
measured **iron-binding capacity** (mg/g peptide) and **DPPH %**, **HRSA EC₅₀** and **SRSA EC₅₀** (radical-scavenging assays).

### 1.1.6 Mung-bean meal antioxidant peptides
**Source:** Sonklin et al. (2021), *Molecules* 26(6):1515, doi:10.3390/molecules26061515 (PMC7999109). Table 1: twelve peptides, each with
DRSA (DPPH), HRSA, SRSA, FRAP and **MCA (metal-chelating activity)**; *ND = no detectable activity*.

**Label derivation (V1, V2) — computed from the tables, not assumed.** A peptide is
* **MC-active** if its chelation measurement is detectable (iron-binding capacity > 0 for Chunkao; MCA not ND for Sonklin);
* **FRS-active** if **any** per-peptide radical-scavenging assay is detectable (DPPH > 0, HRSA or SRSA measurable; FRAP is a reducing-power assay and
  is *not* counted). This is the same *"source measured/indicated an activity"* rule AnOxPePred uses for its FRS label (V3), so the two
  Tier-1 populations are defined consistently.
* **excluded** if it is active in neither.

In [ ]:
# PMC7600763, Table 2 (XML id foods-09-01406-t002)
mungbean_mcp_sequences = fetch_pmc_table_sequences("7600763", "foods-09-01406-t002", expected_n=10)
mungbean_mcp_df = pd.DataFrame({"sequence": mungbean_mcp_sequences})
mungbean_mcp_df["tier"] = "Tier2"
mungbean_mcp_df["source_db"] = "MungBean_MCP_Chunkao2020"
mungbean_mcp_df["evidence_tier"] = EV_PEPTIDE
mungbean_mcp_df["source_ref"] = "doi:10.3390/foods9101406 (Table 2)"

# PMC7999109, Table 1 (XML id molecules-26-01515-t001)
mungbean_meal_mcp_sequences = fetch_pmc_table_sequences("7999109", "molecules-26-01515-t001", expected_n=12)
mungbean_meal_mcp_df = pd.DataFrame({"sequence": mungbean_meal_mcp_sequences})
mungbean_meal_mcp_df["tier"] = "Tier2"
mungbean_meal_mcp_df["source_db"] = "MungBeanMeal_MCP_Sonklin2021"
mungbean_meal_mcp_df["evidence_tier"] = EV_PEPTIDE
mungbean_meal_mcp_df["source_ref"] = "doi:10.3390/molecules26061515 (Table 1)"
print("Mung bean (Chunkao):", len(mungbean_mcp_df), "| Mung-bean meal (Sonklin):", len(mungbean_meal_mcp_df))

In [ ]:
# Per-peptide measurements transcribed from the two tables (re-read from PMC on 2026-10-16).  None = "ND" / not detectable.
# Chunkao 2020, Table 2:  (iron-binding capacity mg/g peptide, DPPH %, HRSA EC50 mM, SRSA EC50 mM)   [GSH ref: DPPH 70.53 %, HRSA 1.92, SRSA 0.92]
CHUNKAO_2020 = {
    "HADAD":    (745.73,  9.06, 1.08, 0.89),
    "PAIDL":    (9058.65, 46.63, 0.09, 0.07),
    "LLLGI":    (288.29, -6.04, 2.75, 5.09),
    "LLLLG":    (940.71, 10.47, 0.86, 0.89),
    "LLGIL":    (2139.68, 81.27, 0.37, 0.36),
    "AIVIL":    (1439.62, 28.80, 0.55, 0.07),
    "AQKIPAGT": (353.62,  4.03, 1.55, 0.57),
    "ILAGPTTI": (349.37, -1.21, 1.54, 0.75),
    "KKGVLGLA": (310.92,  1.01, 1.73, 0.58),
    "RAILTLV":  (597.17,  1.71, 0.90, 0.62),
}
# Sonklin 2021, Table 1:  (DRSA, HRSA, SRSA EC50 mg/mL; FRAP mM Fe2+/mg; MCA EC50 mg/mL)   [GSH ref: 0.03, 0.60, 0.04, 0.61, 0.18]
SONKLIN_2021 = {
    "HC":    (0.29, 0.56, 0.15, 0.71, 1.34),
    "CGN":   (0.28, 0.74, 0.04, 0.08, 0.04),
    "LAN":   (None, 5.31, 1.54, 0.01, 0.36),
    "CTN":   (0.30, 0.64, 0.04, 0.11, 0.04),
    "LAF":   (None, 0.31, 0.02, 0.01, 10.78),
    "CSGD":  (0.30, 0.78, 0.04, 0.16, 0.26),
    "MMGW":  (1.39, 5.96, 1.58, 0.04, 0.77),
    "QFAAD": (None, 2.67, 0.57, 0.01, 1.16),
    "ERF":   (4.87, 2.25, 1.34, 0.01, 0.63),
    "EYW":   (1.32, None, 1.75, 0.06, 0.69),
    "FLQL":  (None, None, None, 0.01, None),      # ND in DRSA, HRSA, SRSA and MCA; FRAP 0.01 (GSH 0.61)
    "QFAW":  (1.67, None, None, 0.03, 0.07),
}

def chunkao_labels(v):
    iron, dpph, hrsa, srsa = v
    return (iron is not None and iron > 0), ((dpph is not None and dpph > 0) or hrsa is not None or srsa is not None)

def sonklin_labels(v):
    drsa, hrsa, srsa, frap, mca = v                  # FRAP is reducing power, NOT radical scavenging -> not used for FRS
    return (mca is not None), any(x is not None for x in (drsa, hrsa, srsa))

MUNG_EVIDENCE = {}
for seq, v in CHUNKAO_2020.items():
    mc, frs = chunkao_labels(v); MUNG_EVIDENCE[seq] = {"paper": "Chunkao2020", "mc": mc, "frs": frs}
for seq, v in SONKLIN_2021.items():
    mc, frs = sonklin_labels(v); MUNG_EVIDENCE[seq] = {"paper": "Sonklin2021", "mc": mc, "frs": frs}

# cross-check: the live-scraped sequence lists must equal the transcribed tables (guards typos AND source changes)
assert set(mungbean_mcp_sequences) == set(CHUNKAO_2020), \
    f"Chunkao scrape != transcription: {set(mungbean_mcp_sequences) ^ set(CHUNKAO_2020)}"
assert set(mungbean_meal_mcp_sequences) == set(SONKLIN_2021), \
    f"Sonklin scrape != transcription: {set(mungbean_meal_mcp_sequences) ^ set(SONKLIN_2021)}"

ev_df = pd.DataFrame([{"sequence": s, **e} for s, e in MUNG_EVIDENCE.items()])
ev_df["verdict"] = np.where(ev_df.mc & ev_df.frs, "dual FRS+MC", np.where(ev_df.mc, "MC only",
                    np.where(ev_df.frs, "FRS only", "NO ACTIVITY -> excluded")))
print(ev_df.groupby(["paper", "verdict"]).size().to_string())

EXCLUDED_NO_ACTIVITY = set(ev_df.loc[~ev_df.mc & ~ev_df.frs, "sequence"])
print("\nExcluded (no detectable activity in the primary paper):", sorted(EXCLUDED_NO_ACTIVITY))
assert EXCLUDED_NO_ACTIVITY == {"FLQL"}, "validated expectation: only FLQL has no detectable activity"
assert all(ev_df.loc[ev_df.sequence != "FLQL", "mc"]), "every other peptide must have detectable chelation"
# sensitivity, for the write-up: how many are FRS-active if DPPH were the ONLY accepted assay?
dpph_only = sum(1 for v in CHUNKAO_2020.values() if v[1] is not None and v[1] > 0) + \
            sum(1 for v in SONKLIN_2021.values() if v[0] is not None)
print(f"FRS-active under the assay-agnostic rule: {int(ev_df.frs.sum())} of {len(ev_df)} | "
      f"under a DPPH-only rule it would be {dpph_only} (reported as a sensitivity, not used)")

### 1.1.7 Multi-AOP
`data/combined/combined_data.csv` has only a **binary** antioxidant label (no FRS / chelator split), so Multi-AOP can feed **Tier 3 only**.

In [ ]:
multiaop_dir = git_clone("https://github.com/CaiJianxiu/Multi-AOP.git", "Multi-AOP")
combined_df = pd.read_csv(f"{multiaop_dir}/data/combined/combined_data.csv")
print(combined_df.shape)                      # expected (5235, 4)
print(combined_df["label"].value_counts())    # expected 2597 positive / 2638 negative

multiaop_positives = combined_df[combined_df["label"] == 1][["SEQUENCE"]].rename(columns={"SEQUENCE": "sequence"})
multiaop_positives["tier"] = "Tier3"
multiaop_positives["source_db"] = "Multi-AOP"
multiaop_positives["evidence_tier"] = EV_DB
multiaop_positives["source_ref"] = "Multi-AOP combined_data.csv (CaiJianxiu/Multi-AOP)"
print(multiaop_positives.shape)               # expected (2597, 3)

### 1.1.8 DFBP
Both official DFBP domains are dead and the journal supplement is methodology only. The download page was archived by the Wayback Machine
as plain text, so the sequences are scraped from the archive. (Network-dependent: if the archive is unavailable this cell fails loudly.)

In [ ]:
raw_url = "https://web.archive.org/web/20220517000000id_/http://cqudfbp.net/download/downloadPage.jsp?pageMark=antioxidative_peptides"
page_text = requests.get(raw_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
pairs = re.findall(r'(DFBPANOX\d+):\s*([A-Z]+)', page_text)
print("Extracted:", len(pairs), "peptides (the archived page's own header says '1032 results')")
assert len(pairs) > 0, "no DFBP peptides extracted -- archive unavailable or page changed"

dfbp_df = pd.DataFrame(pairs, columns=["dfbp_id", "sequence"]).drop_duplicates(subset="sequence")
dfbp_df["tier"] = "Tier3"
dfbp_df["source_db"] = "DFBP"
dfbp_df["evidence_tier"] = EV_DB
dfbp_df["source_ref"] = dfbp_df["dfbp_id"]
print(dfbp_df.shape)                          # expected (1008, 5): 24 duplicate sequences under other DFBP ids removed
dfbp_df.to_csv(f"{RAW_DIR}/DFBP_antioxidative_peptides.csv", index=False)

### 1.1.9 Peptipedia
The Dataset Guide's GitHub repo is only the web-app source. Peptipedia must be queried through its live site, and its *"Include predicted
activities"* toggle must be **OFF** (otherwise > 1,000,000 ML-predicted, not experimentally confirmed, peptides are returned).
The export below assumes: Activity = "Anti oxidative", "Only canonical peptides" = ON, "Include predicted activities" = OFF.
**This one input is manual** — keep `peptipedia_search.csv` in `data/raw/` (its SHA-256 is recorded in the manifest).

In [ ]:
pp_path = find_raw("peptipedia_search.csv")          # manual input; searched in both raw folders (V18)
peptipedia_df = pd.read_csv(pp_path).rename(columns={"Sequence": "sequence"})
PROVENANCE["peptipedia_search_sha256"] = hashlib.sha256(open(pp_path, "rb").read()).hexdigest()

peptipedia_df["seq_len"] = peptipedia_df["sequence"].str.len()
peptipedia_clean = peptipedia_df[(peptipedia_df["seq_len"] >= 3) & (peptipedia_df["seq_len"] <= 50)].copy()
print("Before length filter:", peptipedia_df.shape[0], "| After:", peptipedia_clean.shape[0])   # expected 1096 -> 1059

peptipedia_clean["tier"] = "Tier3"
peptipedia_clean["source_db"] = "Peptipedia"
peptipedia_clean["evidence_tier"] = EV_DB
peptipedia_clean["source_ref"] = "Peptipedia export (Anti oxidative, canonical only, predicted activities OFF)"
peptipedia_clean = peptipedia_clean[["sequence", "tier", "source_db", "evidence_tier", "source_ref"]]
peptipedia_clean.to_csv(f"{PROC}/peptipedia_clean.csv", index=False)

### 1.1.10 BIOPEP-UWM — deliberately not scraped
No bulk download exists (paginated HTML form only). DFBP and Peptipedia both cross-reference BIOPEP-UWM, so its coverage is captured indirectly.
Its activity taxonomy also has no distinct "metal chelating" code (only a generic "binding" code that includes non-antioxidant mineral binding),
so a scraper would carry the same mechanism-conflation risk avoided for DFBP's mineral-binding peptides in §1.4. A documented scope decision.

---
## 1.2 Merge into one master pool — with the primary-evidence exclusion (V1)
A **Tier-1/2 (mechanism-labelled) row** whose primary paper reports *no detectable FRS or MC activity in the cited assays* (`EXCLUDED_NO_ACTIVITY`) is removed.
This is deliberately narrower than "has no antioxidant activity": the rule concerns only the **mechanism labels**. Database-level Tier-3 listings of the same sequence
(DFBP / Peptipedia list `FLQL` as "antioxidative", probably because it was found in an active *fraction*) are left unchanged and remain database-level evidence only.
The removal is logged, not silent.

In [ ]:
cols = ["sequence", "tier", "source_db", "evidence_tier", "source_ref"]
master_df = pd.concat([
    tier1_from_anoxpepred.rename(columns={"Sequence": "sequence"}).assign(source_db="AnOxPePred")[cols],
    tier2_from_anoxpepred.rename(columns={"Sequence": "sequence"}).assign(source_db="AnOxPePred")[cols],
    sunflower_mcp_df[cols], walnut_mcp_df[cols], rice_bran_mcp_df[cols],
    mungbean_mcp_df[cols], mungbean_meal_mcp_df[cols],
    multiaop_positives[cols], dfbp_df[cols], peptipedia_clean[cols],
], ignore_index=True)
master_df["sequence"] = master_df["sequence"].astype(str).str.upper().str.strip()
print("Total raw pool (before exclusion / dedup):", master_df.shape)

excluded_parts = []
# V18: ONE canonical-residue check that covers every source (not only the scraped ones)
bad_res = ~master_df["sequence"].apply(lambda s_: s_ != "" and set(s_) <= VALID_AA)
if bad_res.any():
    excluded_parts.append(master_df[bad_res].assign(rows_removed=1, reason="non-canonical residue or empty sequence"))
master_df = master_df[~bad_res].reset_index(drop=True)
print("rows with non-canonical residues removed:", int(bad_res.sum()))

# V1: remove MECHANISM-LABELLED rows (Tier 1/2) whose primary paper reports no detectable FRS or MC activity; LOG it.
# Tier-3 (database-level) listings of the same sequence are left alone: this is not a claim that the peptide has no antioxidant activity.
is_excl = master_df["sequence"].isin(EXCLUDED_NO_ACTIVITY) & master_df["tier"].isin(["Tier1", "Tier2"])
excluded_parts.append(master_df[is_excl].assign(rows_removed=1,
    reason="no detectable FRS or MC activity in the cited assays (all ND, trace FRAP only); excluded from the mechanism-labelled benchmark"))
master_df = master_df[~is_excl].reset_index(drop=True)

excluded_log = (pd.concat(excluded_parts, ignore_index=True)
                  .groupby(["sequence", "tier", "source_db", "reason"], as_index=False)["rows_removed"].sum())
print("\nExcluded:\n", excluded_log.to_string(index=False) if len(excluded_log) else "(none)")
excluded_log.to_csv(f"{PROC}/excluded_by_primary_evidence.csv", index=False)

print("\nPool after exclusion:", master_df.shape)
print(master_df["tier"].value_counts().to_string())

# V14: the PROVENANCE table keeps every (sequence, tier, source) BEFORE any de-duplication, with the source reference
provenance_df = master_df.copy()
provenance_df["length"] = provenance_df["sequence"].str.len()
provenance_df.to_csv(f"{PROC}/AOP-Provenance.csv", index=False)
print("Saved AOP-Provenance.csv:", provenance_df.shape)
master_df.to_csv(f"{PROC}/merged_raw.csv", index=False)

---
## 1.3 De-duplication
### 1.3.1 Cross-tier duplicates
A sequence present in both a confirmed tier (1/2) and the generic Tier 3 pool keeps only its confirmed label — the mechanism label is strictly
more informative and must not be diluted.

In [ ]:
confirmed_seqs = set(master_df[master_df["tier"].isin(["Tier1", "Tier2"])]["sequence"])
before = len(master_df)
master_df = master_df[~((master_df["tier"] == "Tier3") & (master_df["sequence"].isin(confirmed_seqs)))]
print(f"Removed {before - len(master_df)} Tier3 duplicates of confirmed Tier1/Tier2 sequences")

master_df = master_df.drop_duplicates(subset=["sequence", "tier"])
print(master_df["tier"].value_counts())

### 1.3.2 Global length filter (3–50 aa)
Applied to the **whole** pool (it had originally been applied to Peptipedia only), as the proposal's fixed control variable requires.

In [ ]:
master_df["seq_len"] = master_df["sequence"].str.len()
before = len(master_df)
master_df = master_df[(master_df["seq_len"] >= 3) & (master_df["seq_len"] <= 50)].copy()
print(f"Length filter (3-50 AA): {before} -> {len(master_df)}")
print(master_df["tier"].value_counts())

### 1.3.3 CD-HIT near-duplicate collapse (90 % identity, sequences ≥ 10 aa only)
**How CD-HIT measures identity (V13).** By default (`-G 1`) it is *identical residues ÷ the length of the **shorter** sequence*, and the length-difference and
coverage options (`-s`, `-aS`) default to 0 (CD-HIT user guide, option `-c`). So a peptide that is **contained in** a longer one scores 100 % — expected behaviour, not a bug
(an earlier version of this note called it "impossible"; that was wrong). It does make CD-HIT's identity a poor *redundancy* criterion for short peptides.

CD-HIT also has a minimum length (`-l`, default 10): in an earlier run it did not process the 3-residue peptides at all (1,158 input sequences read from a 1,639-row table).
CD-HIT is therefore applied **only to sequences ≥ 10 aa**, with its standard parameters (`-n 5 -l 10`); shorter sequences rely on the exact-match de-duplication of §1.3.1–1.3.2.
Near-duplicates among short peptides are handled for *splitting* in §1.6.2, with an explicit identity definition (identical residues ÷ the **longer** length, via edit distance).

In [ ]:
!apt-get install -y cd-hit -q

def run_cdhit_safe(df, tier_name, identity=0.90, min_cdhit_length=10):
    """
    Deduplicate one tier at a given identity threshold, but only run CD-HIT on
    sequences >= min_cdhit_length (CD-HIT is unreliable below ~10 AA -- see note above).
    Shorter sequences pass through as-is (already exact-deduplicated in 1.3.1/1.3.2).
    """
    subset = df[df["tier"] == tier_name]
    short_seqs = subset[subset["sequence"].str.len() < min_cdhit_length]["sequence"].tolist()
    long_subset = subset[subset["sequence"].str.len() >= min_cdhit_length]

    long_kept = []
    if len(long_subset) > 0:
        fasta_path = f"/content/{tier_name}_long.fasta"
        with open(fasta_path, "w") as f:
            for i, row in long_subset.iterrows():
                f.write(f">{tier_name}_{i}\n{row['sequence']}\n")
        out_path = f"/content/{tier_name}_long_cdhit"
        subprocess.run([
            "cd-hit", "-i", fasta_path, "-o", out_path,
            "-c", str(identity), "-n", "5", "-l", str(min_cdhit_length),
            "-M", "0", "-d", "0"
        ], check=True)
        long_kept = [line.strip() for line in open(out_path) if not line.startswith(">")]

    return list(set(short_seqs)) + long_kept


tier1_final = run_cdhit_safe(master_df, "Tier1")
tier2_final = run_cdhit_safe(master_df, "Tier2")
tier3_final = run_cdhit_safe(master_df, "Tier3")

print("Tier1 final:", len(tier1_final))
print("Tier2 final:", len(tier2_final))
print("Tier3 final:", len(tier3_final))

---
## 1.4 Data sufficiency — Tier 2
The proposal set a planning target of 150 Tier-2 peptides. Recovery routes examined (documented, not omitted): AODB (a real database, Deng et al.
2023, but its portal is down and a downstream reuse carries only a binary label); DFBP "mineral-binding" peptides (checked below: no overlap with the
confirmed antioxidants, and they are casein phosphopeptides — a different mechanism); the five literature sources of §1.1.2–1.1.6.

**The count is reported as it is.** No peptide is added or relabelled to hit a number (V10). The cell after the DFBP check prints the Tier-2 total and
how much of it rests on per-peptide evidence versus candidate (complex-level) evidence.

In [ ]:
raw_url = "https://web.archive.org/web/20220517000000id_/http://www.cqudfbp.net/download/downloadPage.jsp?pageMark=mineral_binding_peptides"
page_text = requests.get(raw_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text

pairs = re.findall(r'(DFBP\w+\d+):\s*([A-Z]+)', page_text)
mineral_seqs = set(seq for _, seq in pairs)
antioxidant_seqs = set(dfbp_df["sequence"])

overlap = mineral_seqs & antioxidant_seqs
print("Mineral-binding peptides found:", len(mineral_seqs))
print("Confirmed BOTH antioxidant AND mineral-binding:", len(overlap))

# Zero overlap. Inspecting the mineral-binding sequences shows they are casein phosphopeptides
# (calcium-absorption enhancers) -- a different biological mechanism entirely from antioxidant
# metal-chelation. Correctly excluded: including them would have been a mechanism-conflation error.

In [ ]:
t2 = master_df[(master_df["tier"] == "Tier2") & (master_df["sequence"].isin(tier2_final))].drop_duplicates("sequence")
print(t2.groupby(["source_db", "evidence_tier"]).size().to_string())
n_t2 = len(tier2_final)
n_validated = int((t2["evidence_tier"] != EV_COMPLEX).sum())
print(f"\nTier 2 after validation and de-duplication: {n_t2}   (planning target in the proposal: 150)")
print(f"  of which per-peptide evidence: {n_validated}   |   candidate (complex/hydrolysate-level): {n_t2 - n_validated}")
if n_t2 < 150:
    print(f"  -> {150 - n_t2} below the planning target. Reported as-is; nothing is added or relabelled to reach it.")

---
## 1.5 AOP-BenchPos (S1, intermediate)
One row per (sequence, tier). A sequence confirmed in **both** a Tier-1 and a Tier-2 source legitimately appears once in each tier (de-duplication is per tier);
`dual_mechanism` flags it so it is never mistaken for a de-duplication failure. S1 is kept as a reference; **S2 (§1.7) is the dataset used downstream.**

In [ ]:
def build_final_tier_df(seq_list, tier_name):
    tier_rows = master_df[(master_df["tier"] == tier_name) & (master_df["sequence"].isin(seq_list))]
    tier_rows = tier_rows.drop_duplicates(subset="sequence")
    return tier_rows[["sequence", "tier", "source_db", "evidence_tier"]]

aop_benchpos = pd.concat([
    build_final_tier_df(tier1_final, "Tier1"),
    build_final_tier_df(tier2_final, "Tier2"),
    build_final_tier_df(tier3_final, "Tier3"),
], ignore_index=True)

dual_seqs = set(aop_benchpos.loc[aop_benchpos["sequence"].duplicated(keep=False), "sequence"])
aop_benchpos["dual_mechanism"] = aop_benchpos["sequence"].isin(dual_seqs)

print(aop_benchpos.shape, "rows =", aop_benchpos["sequence"].nunique(), "unique sequences")
print(aop_benchpos["tier"].value_counts().to_string())
print("Dual-mechanism sequences present in both Tier 1 and Tier 2 sources:", len(dual_seqs))
print("\nSource breakdown:")
print(aop_benchpos.groupby(["tier", "source_db"]).size().to_string())

In [ ]:
# V19 -- label-loss audit. Per-tier CD-HIT could drop a sequence from one tier while it survives in the other, silently losing a dual label.
prov = provenance_df[provenance_df["tier"].isin(["Tier1", "Tier2"]) & provenance_df["length"].between(3, 50)]
kept = set(zip(aop_benchpos["tier"], aop_benchpos["sequence"]))
lost = sorted({(t, q) for t, q in zip(prov["tier"], prov["sequence"]) if (t, q) not in kept})
final_seqs = set(aop_benchpos["sequence"])
lost_but_retained_elsewhere = [(t, q) for t, q in lost if q in final_seqs]
print(f"(tier, sequence) pairs with exact source evidence that were removed by de-duplication: {len(lost)} "
      f"(<10 aa: {sum(len(q) < 10 for _, q in lost)}, >=10 aa: {sum(len(q) >= 10 for _, q in lost)})")
print(f"  ...of which the sequence survives under ANOTHER tier (a dual label would have been lost): {len(lost_but_retained_elsewhere)}")
assert not lost_but_retained_elsewhere, f"dual label lost to per-tier de-duplication: {lost_but_retained_elsewhere[:5]}"
print("OK: de-duplication did not destroy any dual label.")

### 1.5b Evidence conflicts — Tier 3 versus AnOxPePred's experimentally validated negatives (V20)
AnOxPePred's authors report **218 "experimentally-validated non-antioxidant" peptides** (FRS = 0 and Chelator = 0, labelled from the measurements in each source study).
Tier 3 is built from three *databases* (Multi-AOP, DFBP, Peptipedia) whose "antioxidant" label is a database-level binary label. The two can **contradict each other for the same sequence.**

This cell measures how often, and applies one explicit policy (`CONFLICT_POLICY`):

* `"exclude"` *(default)* — a Tier-3 sequence that AnOxPePred reports as experimentally inactive is **removed from the Tier-3 (GEN) pool** and written to `AOP-Conflicts.csv`.
  Rationale: Tier 3 is meant to be *confirmed* antioxidants; a sequence contradicted by a per-peptide negative is not confirmed. (The databases may be right — e.g. they may list every peptide
  found in an antioxidant *hydrolysate* — but that is exactly the kind of weaker, fraction-level evidence Tier 3 cannot rely on.)
* `"keep"` — leave them in GEN but still flag them (`db_conflict`) so any analysis can drop them.

A conflict between AnOxPePred's negatives and a **Tier-1/2** sequence would be a contradiction *inside* the confirmed set; the cell lists any and stops, rather than choosing silently.

In [ ]:
CONFLICT_POLICY = "exclude"          # "exclude" (default) or "keep"; see the text above

inactive_set = set(anox_inactive["sequence"])
raw_t3   = set(provenance_df.loc[provenance_df["tier"] == "Tier3", "sequence"])                 # every Tier-3 listing, BEFORE de-duplication
raw_t12  = set(provenance_df.loc[provenance_df["tier"].isin(["Tier1", "Tier2"]), "sequence"])
t3_final_set = set(tier3_final)

conflict_t12_raw = sorted(inactive_set & raw_t12)
conflict_raw_t3  = sorted(inactive_set & raw_t3)          # listed "antioxidant" by a database anywhere in the raw pool
conflict_t3      = sorted(inactive_set & t3_final_set)    # ... and still present in the RETAINED Tier 3
print(f"AnOxPePred experimentally-validated non-antioxidant sequences : {len(inactive_set)}")
print(f"  also in the confirmed Tier 1/2 raw pool                     : {len(conflict_t12_raw)}")
print(f"  listed by a Tier-3 database in the RAW pool                 : {len(conflict_raw_t3)}")
print(f"  ...of which still in the RETAINED Tier 3                    : {len(conflict_t3)}  = {len(conflict_t3) / max(len(t3_final_set), 1):.1%} of Tier 3")
print(f"  ...listed but dropped by the length filter / de-duplication : {len(conflict_raw_t3) - len(conflict_t3)}  (still NOT clean negatives)")
assert not conflict_t12_raw, f"contradiction inside the confirmed set -- decide manually: {conflict_t12_raw[:10]}"

src_of = provenance_df[provenance_df["sequence"].isin(conflict_raw_t3) & (provenance_df["tier"] == "Tier3")]
print("\nTier-3 databases that list the conflicting sequences (a sequence can be listed by several):")
print(src_of.groupby("source_db")["sequence"].nunique().to_string())
ref = df[df["Sequence"].isin(conflict_raw_t3)]
print("\nreferences AnOxPePred reports for these inactive peptides (top 5):")
print(ref["Source"].value_counts().head(5).to_string())

conflicts_df = pd.DataFrame({"sequence": conflict_raw_t3})
conflicts_df["anox_label"] = "FRS=0, Chelator=0 (experimentally validated non-antioxidant)"
conflicts_df["listed_antioxidant_by"] = conflicts_df["sequence"].map(
    src_of.groupby("sequence")["source_db"].agg(lambda x: ";".join(sorted(set(x)))))
conflicts_df["retained_in_tier3"] = conflicts_df["sequence"].isin(t3_final_set)
conflicts_df["policy"] = CONFLICT_POLICY
conflicts_df.to_csv(f"{PROC}/AOP-Conflicts.csv", index=False)

if CONFLICT_POLICY == "exclude":
    n_before = len(aop_benchpos)
    aop_benchpos = aop_benchpos[~((aop_benchpos["tier"] == "Tier3") & aop_benchpos["sequence"].isin(conflict_t3))].reset_index(drop=True)
    tier3_final = [q for q in tier3_final if q not in set(conflict_t3)]
    print(f"\nPOLICY 'exclude': removed {n_before - len(aop_benchpos)} Tier-3 rows -> Tier 3 now {len(tier3_final)} sequences "
          f"(was {len(t3_final_set)}). All {len(conflict_raw_t3)} raw-pool conflicts are listed in AOP-Conflicts.csv.")
else:
    print("\nPOLICY 'keep': Tier 3 unchanged; the conflicting sequences stay in GEN and are flagged `db_conflict` in S2.")
CONFLICT_SET  = set(conflict_t3)          # retained Tier-3 conflicts (what the policy acts on; also S2 `db_conflict`)
DB_LISTED_SET = set(conflict_raw_t3)      # raw-pool conflicts (what makes an inactive peptide NOT 'clean')

---
## 1.6 Leak-safe homology grouping and the 2-fold split
**Goal:** split the Tier 1/2 sequences so that no two sequences that are too similar ever fall on opposite sides of the split — otherwise a retrieval
model or probe can score well by *remembering* a near-duplicate of a test sequence.

**Tool note.** The proposal names GraphPart/HESTIA; CD-HIT (already installed and debugged for short peptides in §1.3) is used instead, at 40 % identity.

### 1.6.1 CD-HIT at 40 % identity (sequences ≥ 10 aa only)
Bug found earlier and fixed: with `-l 3`, CD-HIT silently dropped all 481 three-residue peptides (it reported 1,158 input sequences for a 1,639-row table).
CD-HIT is therefore trusted only for ≥ 10 aa.

**Important (V7): what happens to the shorter peptides.** They cannot be clustered by CD-HIT, so each gets its *own singleton id*. That avoids
*wrongly merging* dissimilar peptides, but it does **not** protect against leakage: two near-identical short peptides (e.g. a one-residue variant)
receive different ids and can land in different folds. The earlier note calling this "the conservative choice" was true only for merging, not for leakage.
§1.6.2 closes that gap.

In [ ]:
def cluster_40_safe(df, min_cluster_length=10):
    """CD-HIT (40 % identity) for sequences >= min_cluster_length; every shorter sequence gets its own singleton id.
    Singleton ids are NOT a leakage guard -- near-duplicate short peptides are linked later (Section 1.6.2)."""
    df = df.reset_index(drop=True)
    long_mask = df["sequence"].str.len() >= min_cluster_length
    short_mask = ~long_mask

    fasta_path = "/content/aop_benchpos_40_long.fasta"
    output_prefix = "/content/aop_benchpos_40_long"
    with open(fasta_path, "w") as f:
        for i in df[long_mask].index:
            f.write(f">seq{i}\n{df.loc[i, 'sequence']}\n")

    subprocess.run([
        "cd-hit", "-i", fasta_path, "-o", output_prefix,
        "-c", "0.40", "-n", "2", "-l", str(min_cluster_length), "-M", "0", "-d", "0"
    ], check=True, capture_output=True, text=True)

    cluster_map, current_cluster = {}, None
    with open(output_prefix + ".clstr") as f:
        for line in f:
            if line.startswith(">Cluster"):
                current_cluster = int(line.strip().split()[-1])
            else:
                m = re.search(r">seq(\d+)\.\.\.", line)
                if m:
                    cluster_map[int(m.group(1))] = current_cluster

    next_id = max(cluster_map.values(), default=-1) + 1          # offset past the CD-HIT ids: no collision
    for i in df[short_mask].index:
        cluster_map[i] = next_id
        next_id += 1

    df["cluster_40"] = df.index.map(cluster_map)
    assert df["cluster_40"].isna().sum() == 0, "clustering left unassigned sequences -- investigate before continuing"
    return df

aop_benchpos = cluster_40_safe(aop_benchpos)
print("Rows:", len(aop_benchpos), "| distinct cluster_40 ids:", aop_benchpos["cluster_40"].nunique(),
      "| rows < 10 aa (singleton ids):", int((aop_benchpos["sequence"].str.len() < 10).sum()))

### 1.6.2 Homology groups, then folds (V7)
**Group definition.** Two Tier-1/2 sequences belong to the same **homology group** if any of the following holds (transitively, via union–find):
1. they are the *same sequence* (a dual peptide has a Tier-1 row and a Tier-2 row, possibly with two cluster ids);
2. they share a `cluster_40` id (CD-HIT, ≥ 10 aa);
3. they are **near-duplicates**: **normalised edit similarity ≥ 70 %**, i.e. `1 − Levenshtein ÷ longer length ≥ 0.70`, where **at least one of the two peptides is ≥ 4 aa**
   (so a 3-aa peptide *is* linked to a 4-aa peptide that differs by one residue, 75 %; two 3-aa peptides that differ by one residue, 67 %, are **not**). For 4–6-mers this
   means "at most one edit", for 7–9-mers "at most two".

   **What this rule is and is not.** The 70 % number is the partition threshold AnOxPePred's authors chose, but they compute identity with a *Needleman–Wunsch alignment*
   (identical = 1, dissimilar = 0, gap −10, end gaps free), which is a different quantity. This is **our own rule**; it is not the same method as AnOxPePred's, and it is **not**
   the proposal's 40 % identity separation. Only the guarantees printed by the audit in §1.6.2b are claimed.

**Folds.** Whole groups go to one fold. Assignment is a **deterministic greedy stratified partition** (no seed): groups that contain MC-active sequences go first, each to the fold currently holding fewer MC-active
sequences; all other groups then go to the fold with fewer sequences in total. This keeps both the small MC class *and* the fold sizes balanced, which a random split does not guarantee.

**Verification, independent of the construction:** after assignment the near-duplicate search is **re-run from scratch** and must find zero pairs
spanning the folds; and no sequence, no `cluster_40` id and no `homology_group` may span both folds.

In [ ]:
NEAR_IDENT   = 0.70        # AnOxPePred's own homology-partition threshold (Olsen et al. 2020)
NEAR_MIN_LEN = 4

def _within_edit(a, b, k):
    """True iff Levenshtein(a, b) <= k  (banded dynamic programme, exact)."""
    la, lb = len(a), len(b)
    if abs(la - lb) > k:
        return False
    INF = k + 1
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [INF] * lb
        lo, hi = max(1, i - k), min(lb, i + k)
        for j in range(lo, hi + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
        if min(prev) > k:
            return False
    return prev[lb] <= k

def is_near_duplicate(a, b):
    la, lb = len(a), len(b)
    if max(la, lb) < NEAR_MIN_LEN:           # at least one of the pair must be >= 4 aa (3-aa <-> 3-aa has k = 0 anyway)
        return False
    k = int(max(la, lb) * (1 - NEAR_IDENT) + 1e-9)      # largest edit distance that still keeps identity >= 70 %
    return k >= 1 and _within_edit(a, b, k)

def near_duplicate_pairs(seqs):
    seqs = sorted(set(seqs), key=lambda s: (len(s), s))
    pairs = []
    for i, a in enumerate(seqs):
        la = len(a)
        for b in seqs[i + 1:]:
            if len(b) - la > int(len(b) * (1 - NEAR_IDENT) + 1e-9):     # sorted by length: nothing further can match
                break
            if is_near_duplicate(a, b):
                pairs.append((a, b))
    return pairs

class UnionFind:
    def __init__(self): self.p = {}
    def find(self, x):
        self.p.setdefault(x, x)
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)         # deterministic root = smallest key

def build_homology_groups(tier12_rows):
    """tier12_rows: rows of Tier1/Tier2 with columns sequence, cluster_40.  Returns {sequence: group_id} (ids are compact ints)."""
    uf = UnionFind()
    seqs = sorted(set(tier12_rows["sequence"]))
    for s in seqs: uf.find(s)
    for _, grp in tier12_rows.groupby("cluster_40"):            # (1)+(2): same sequence / same CD-HIT or singleton id
        members = sorted(set(grp["sequence"]))
        for s in members[1:]: uf.union(members[0], s)
    for _, grp in tier12_rows.groupby("sequence"):              # a dual peptide's two rows may carry two cluster ids
        ids = sorted(set(grp["cluster_40"]))
        if len(ids) > 1:
            by_id = [sorted(set(tier12_rows.loc[tier12_rows.cluster_40 == c, "sequence"])) for c in ids]
            for m in by_id[1:]: uf.union(by_id[0][0], m[0])
    n_cluster_links = len({uf.find(s) for s in seqs})
    pairs = near_duplicate_pairs(seqs)                          # (3)
    for a, b in pairs: uf.union(a, b)
    roots = sorted({uf.find(s) for s in seqs})
    rid = {r: i for i, r in enumerate(roots)}
    return {s: rid[uf.find(s)] for s in seqs}, pairs, n_cluster_links

def assign_folds_greedy(groups, mc_seqs, n_folds=2):
    """groups: {sequence: group_id}; mc_seqs: set of MC-active sequences.  Deterministic, no seed.
    Phase A balances the MC-active count (then total size) over the groups that contain MC-active sequences;
    phase B places every remaining group so that the TOTAL sizes end up balanced."""
    g_size, g_mc = {}, {}
    for s_, g_ in groups.items():
        g_size[g_] = g_size.get(g_, 0) + 1
        g_mc[g_] = g_mc.get(g_, 0) + (s_ in mc_seqs)
    with_mc = sorted([g_ for g_ in g_size if g_mc[g_] > 0], key=lambda g_: (-g_mc[g_], -g_size[g_], g_))
    rest    = sorted([g_ for g_ in g_size if g_mc[g_] == 0], key=lambda g_: (-g_size[g_], g_))
    load = [[0, 0] for _ in range(n_folds)]                     # [mc, total] per fold
    fold_of = {}
    for g_ in with_mc:
        f = min(range(n_folds), key=lambda k: (load[k][0], load[k][1], k))
        fold_of[g_] = f; load[f][0] += g_mc[g_]; load[f][1] += g_size[g_]
    for g_ in rest:
        f = min(range(n_folds), key=lambda k: (load[k][1], k))
        fold_of[g_] = f; load[f][1] += g_size[g_]
    return fold_of

tier12_mask = aop_benchpos["tier"].isin(["Tier1", "Tier2"])
tier12 = aop_benchpos[tier12_mask]
mc_seqs = set(tier12.loc[tier12["tier"] == "Tier2", "sequence"])

seq_group, near_pairs, n_cluster_only_groups = build_homology_groups(tier12[["sequence", "cluster_40"]])
fold_of_group = assign_folds_greedy(seq_group, mc_seqs, n_folds=2)

aop_benchpos["homology_group"] = aop_benchpos["sequence"].map(seq_group)
aop_benchpos["lso_fold"] = aop_benchpos["homology_group"].map(fold_of_group)
aop_benchpos.loc[~tier12_mask, ["homology_group", "lso_fold"]] = np.nan

n_seq = len(seq_group)
print(f"Tier1/2 unique sequences: {n_seq} | groups from cluster_40 + identical sequence: {n_cluster_only_groups} "
      f"| final homology groups (after near-duplicate links): {len(set(seq_group.values()))}")
print(f"near-duplicate pairs (identity >= {NEAR_IDENT:.0%}, at least one >= {NEAR_MIN_LEN} aa): {len(near_pairs)}")
_gs = pd.Series(list(seq_group.values())).value_counts()
print(f"largest homology group: {int(_gs.iloc[0])} sequences ({_gs.iloc[0] / n_seq:.1%} of the set) | groups of size 1: {int((_gs == 1).sum())}")
fold_tab = (pd.DataFrame({"sequence": list(seq_group), "fold": [fold_of_group[g] for g in seq_group.values()],
                          "MC": [s in mc_seqs for s in seq_group]})
              .groupby("fold").agg(sequences=("sequence", "size"), MC_active=("MC", "sum")))
print(fold_tab.to_string())

# ---- verification, independent of the construction --------------------------------------------------
a = aop_benchpos[tier12_mask]
assert a.groupby("sequence")["lso_fold"].nunique().max() == 1, "a sequence spans both folds"
assert a.groupby("cluster_40")["lso_fold"].nunique().max() == 1, "a cluster_40 id spans both folds"
assert a.groupby("homology_group")["lso_fold"].nunique().max() == 1, "a homology group spans both folds"
fold_by_seq = a.drop_duplicates("sequence").set_index("sequence")["lso_fold"]
recheck = near_duplicate_pairs(fold_by_seq.index)                   # recomputed from scratch
crossing = [(x, y) for x, y in recheck if fold_by_seq[x] != fold_by_seq[y]]
assert not crossing, f"{len(crossing)} near-duplicate pairs still span the folds, e.g. {crossing[:3]}"
print(f"\nOK: 0 sequences, 0 cluster_40 ids, 0 homology groups and 0 of {len(recheck)} near-duplicate pairs span both folds.")

### 1.6.2b Train–test similarity audit (what the split does and does not guarantee)
For every fold-1 sequence the cell finds its **most similar fold-0 sequence** and reports that identity, by length class. Identity is `1 − Levenshtein ÷ longer length` (our own measure, §1.6.2).

What is guaranteed, stated precisely:
* **Any pair with at least one peptide ≥ 4 aa and similarity ≥ 70 %** is in the same fold — *asserted*, recomputed from scratch over every fold-0 × fold-1 pair. (This includes 3-aa ↔ 4-aa pairs.)
* **≥ 10 aa:** additionally, no two sequences in the same CD-HIT 40 % cluster (CD-HIT measures identity against the *shorter* sequence, so this is looser than the 70 % rule).
* **3-aa ↔ 3-aa pairs: no guarantee.** One substitution is 67 % identity; linking them would join 229 of the 259 tripeptides into one giant component and make a balanced split impossible
  (AnOxPePred's authors report the same effect). The number of cross-fold 1-substitution tripeptide pairs is printed rather than hidden.
* **Not claimed:** equivalence with the proposal's 40 % identity separation, or with AnOxPePred's alignment-based identity.

Consequence for the experiments: repeat them on `homology_resolvable` (≥ 4 aa) with the 3-aa peptides **removed from training, validation and the retrieval gallery**.

In [ ]:
def edit_distance(a, b):
    la, lb = len(a), len(b)
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        for j in range(1, lb + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return prev[lb]

f_by_seq = aop_benchpos[tier12_mask].drop_duplicates("sequence").set_index("sequence")["lso_fold"]
f0 = [q for q, f in f_by_seq.items() if f == 0]
f1 = [q for q, f in f_by_seq.items() if f == 1]
rows, violations, tri_cross = [], 0, 0
for q in f1:
    best_id = 0.0
    for t in f0:
        la, lb = len(q), len(t)
        if min(la, lb) / max(la, lb) < 0.5:           # identity cannot reach 0.5 when the shorter is < half the longer
            continue
        ed = edit_distance(q, t)
        ident = 1 - ed / max(la, lb)
        best_id = max(best_id, ident)
        if max(la, lb) >= NEAR_MIN_LEN and ed <= int(max(la, lb) * (1 - NEAR_IDENT) + 1e-9):
            violations += 1
        if la == lb == 3 and ed == 1:
            tri_cross += 1
    rows.append((q, len(q), best_id))
au = pd.DataFrame(rows, columns=["sequence", "length", "max_identity_to_other_fold"])
au["class"] = np.where(au["length"] == 3, "3 aa", np.where(au["length"] < 10, "4-9 aa", ">=10 aa"))
tab = au.groupby("class")["max_identity_to_other_fold"].agg(
    n="size", median="median",
    ge_70=lambda x: int((x >= NEAR_IDENT).sum()), ge_60=lambda x: int((x >= 0.60).sum()), ge_50=lambda x: int((x >= 0.50).sum()))
print("fold-1 sequences vs their most similar fold-0 sequence (identity = 1 - Levenshtein / longer length)")
print(tab.round(3).to_string())
print(f"\ncross-fold pairs with identity >= {NEAR_IDENT:.0%} where at least one peptide is >= {NEAR_MIN_LEN} aa: {violations}   (guarantee: 0)")
print(f"cross-fold 1-substitution TRIPEPTIDE pairs (67 % identity, not preventable): {tri_cross}")
assert violations == 0, "the split violates its own stated guarantee"
n_tri = int((aop_benchpos[tier12_mask].drop_duplicates("sequence")["sequence"].str.len() == 3).sum())
n_all12 = int(aop_benchpos[tier12_mask]["sequence"].nunique())
print(f"tripeptides: {n_tri} of {n_all12} mechanism-active sequences ({n_tri / n_all12:.0%}) -> the homology guarantee does not cover them")
AUDIT = {"cross_fold_ge70_len_ge4": int(violations), "cross_fold_1sub_tripeptide_pairs": int(tri_cross),
         "tripeptide_fraction": round(n_tri / n_all12, 4), "table": tab.round(3).reset_index().to_dict(orient="records")}

### 1.6.3 Tier 3 — two evaluation designs (both kept; Design A is the reported one)
Tier 3 is **never used to train** (only Tier 1/2 positive pairs shape the embedding space). Its role is evaluation: each Tier-3 query is compared against fixed
prototype vectors, not against other Tier-3 peptides, so withholding sources would discard data without changing any single query's result.

* **Design A (reported):** merge all Tier-3 sequences as one query pool and report per-`source_db` breakdowns.
* **Design B (supplementary):** rotating leave-source-out, kept for traceability to the proposal wording. It prunes leakage with `cluster_40` only (short peptides are
  singletons), so its "dropped" counts are a **lower bound**; the numbers below are printed from this run, not quoted.

In [ ]:
tier3_df = aop_benchpos[aop_benchpos["tier"] == "Tier3"]
t3_seqs = set(tier3_df["sequence"])
membership = provenance_df[provenance_df["sequence"].isin(t3_seqs)]
print("Tier 3 evaluation pool:", len(t3_seqs), "unique sequences")
print("\nRETAINED-source attribution (the `source_db` shown in S1/S2 -- NOT complete membership):")
print(tier3_df.groupby("source_db").size().to_string())
print("\nFULL membership (a sequence can be listed by several databases):")
print(membership.groupby("source_db")["sequence"].nunique().to_string())
multi = membership.groupby("sequence")["source_db"].nunique()
print(f"\nsequences listed by more than one database: {int((multi > 1).sum())} of {len(t3_seqs)}")
print("-> report per-database results with FULL membership (S2 column `all_sources`), never the retained attribution.")

def build_lso_split(df, held_out_source):
    # Exploratory only: uses the RETAINED attribution and cluster_40 (short peptides are singletons), so 'dropped' is a lower bound.
    test_mask = df["source_db"] == held_out_source
    test_clusters = set(df.loc[test_mask, "cluster_40"])
    leak_mask = (~test_mask) & (df["cluster_40"].isin(test_clusters))
    train_mask = (~test_mask) & (~leak_mask)
    return train_mask, test_mask, leak_mask

print(f"\n{'Held-out source (exploratory)':<30} {'Train':>7} {'Test':>7} {'Dropped':>9}")
for held_out in sorted(tier3_df["source_db"].unique()):
    train_mask, test_mask, leak_mask = build_lso_split(tier3_df, held_out)
    print(f"{held_out:<30} {train_mask.sum():>7} {test_mask.sum():>7} {leak_mask.sum():>9}")

### 1.6.4 Save S1

In [ ]:
s1_path, s1_bak = f"{PROC}/AOP-BenchPos.csv", f"{PROC}/AOP-BenchPos.previous.csv"
if os.path.exists(s1_path) and not os.path.exists(s1_bak):
    import shutil; shutil.copy(s1_path, s1_bak); print("backed up previous S1 ->", s1_bak)
aop_benchpos.to_csv(s1_path, index=False)
print("Saved AOP-BenchPos.csv (S1). Columns:", aop_benchpos.columns.tolist())

---
## 1.7 S2 — the multi-label dataset used downstream
S1 keeps a dual-mechanism peptide as two rows (one per tier). S2 has **one row per unique sequence** with a label set drawn from {FRS, MC, GEN}; a dual peptide is a
legitimate member of both classes.

**Label rules**
* Tier 1 row → `FRS`; Tier 2 row → `MC`; Tier 3 row → `GEN`.
* A Tier-2 mung-bean / mung-bean-meal peptide additionally gets `FRS` **only if the per-peptide table shows detectable radical scavenging** (`MUNG_EVIDENCE[...]["frs"]`, §1.1.6) —
  no longer a blanket reclassification (V2).
* `sample_weight = 0.5` for sequences whose **only** evidence is complex/hydrolysate-level (sunflower, walnut), else 1.0.

**New provenance columns** (appended after the original eight, so downstream code is unaffected):
`homology_group` (use this, not `cluster_40`, for any grouped cross-validation), `length`, `source_group` (AnOxPePred / external_literature / tier3_db),
`mc_evidence` (`validated` / `candidate`), `frs_evidence` (`source_reported` / `per_peptide_radical_assay`).

**Backup first:** if an S2 file already exists, it is copied to `AOP-BenchPos_S2.previous.csv` before anything is overwritten, and a row-level diff is written to `s2_change_log.csv`.

In [ ]:
TIER2LAB = {"Tier1": "FRS", "Tier2": "MC", "Tier3": "GEN"}
aop_benchpos["mech"] = aop_benchpos["tier"].map(TIER2LAB)

extra_frs = {s for s, e in MUNG_EVIDENCE.items() if e["frs"]}            # validated per-peptide radical scavenging (V2)

def weight_of(evidence_set):
    non_complex = evidence_set - {EV_COMPLEX}
    return 1.0 if non_complex else 0.5

# V14: complete source membership from the provenance table (before de-duplication)
_grp = provenance_df.groupby("sequence")["source_db"]
all_src = _grp.agg(lambda x: ";".join(sorted(set(x)))).to_dict()
n_src   = _grp.agg(lambda x: len(set(x))).to_dict()

records = []
for seq, g in aop_benchpos.groupby("sequence"):
    labels = set(g["mech"])
    if seq in extra_frs and "MC" in labels:
        labels.add("FRS")
    ev   = set(g["evidence_tier"])
    srcs = set(g["source_db"])
    folds = sorted(set(g["lso_fold"].dropna().tolist()))
    assert len(folds) <= 1, f"{seq}: sequence spans >1 fold {folds}"
    grp = sorted(set(g["homology_group"].dropna().astype(int).tolist()))
    assert len(grp) <= 1, f"{seq}: sequence in >1 homology group {grp}"
    if "GEN" in labels:
        src_group = "tier3_db"
    elif "AnOxPePred" in srcs:
        src_group = "AnOxPePred"
    else:
        src_group = "external_literature"
    mc_ev  = ("validated" if ev & {EV_SOURCE, EV_PEPTIDE} else "candidate") if "MC" in labels else ""
    frs_ev = (("source_reported" if EV_SOURCE in ev else "per_peptide_radical_assay") if "FRS" in labels else "")
    # V15: strict = every mechanism label this sequence carries is supported at the level of the individual peptide
    strict = (None if "GEN" in labels else (mc_ev != "candidate"))
    records.append({
        "sequence": seq,
        "labels": ";".join(sorted(labels)),                 # multi-hot, e.g. "FRS;MC"
        "n_labels": len(labels),
        "source_db": ";".join(sorted(srcs)),
        "evidence_tier": ";".join(sorted(ev)),
        "cluster_40": int(g["cluster_40"].min()),           # representative CD-HIT / singleton id
        "lso_fold": (folds[0] if folds else None),          # GEN (eval-only) carries no fold
        "sample_weight": weight_of(ev),
        "homology_group": (grp[0] if grp else None),
        "length": len(seq),
        "source_group": src_group,
        "mc_evidence": mc_ev,
        "frs_evidence": frs_ev,
        "strict_evidence": strict,
        "homology_resolvable": len(seq) >= NEAR_MIN_LEN,
        "all_sources": all_src.get(seq, ""),
        "n_sources": n_src.get(seq, 0),
        "db_conflict": seq in CONFLICT_SET,
    })

aop_benchpos_s2 = pd.DataFrame(records)
aop_benchpos_s2["homology_group"] = aop_benchpos_s2["homology_group"].astype("Int64")
aop_benchpos_s2["strict_evidence"] = aop_benchpos_s2["strict_evidence"].astype("boolean")
print("S2 unique sequences:", len(aop_benchpos_s2))
print(aop_benchpos_s2["labels"].value_counts().to_string())
for m in ["FRS", "MC", "GEN"]:
    print(f"  {m}-active: {int(aop_benchpos_s2['labels'].str.contains(m).sum())}")

In [ ]:
s2 = aop_benchpos_s2
mech_s2 = s2[s2["labels"].str.contains("FRS|MC")]
is_gen = s2["labels"] == "GEN"

# ---- invariants (every assertion must pass) ----------------------------------------------------------
assert s2["sequence"].is_unique, "S2 must be one row per unique sequence"
assert s2["labels"].str.split(";").explode().isin({"FRS", "MC", "GEN"}).all()
assert int(s2["labels"].str.contains("MC").sum()) == len(tier2_final), "MC-active must equal the validated Tier-2 count"
assert int(s2["labels"].str.contains("GEN").sum()) == len(tier3_final), "GEN must equal Tier 3"
assert s2.loc[s2["sequence"].isin(EXCLUDED_NO_ACTIVITY), "labels"].isin(["GEN"]).all(), "an excluded peptide carries a mechanism label in S2"
assert s2.loc[is_gen, "lso_fold"].isna().all() and s2.loc[is_gen, "homology_group"].isna().all(), "GEN must carry no fold / group"
assert s2.loc[~is_gen, "lso_fold"].notna().all() and s2.loc[~is_gen, "homology_group"].notna().all(), "every FRS/MC sequence needs a fold and group"
assert set(s2["sample_weight"].unique()) <= {0.5, 1.0}
assert mech_s2.groupby("cluster_40")["lso_fold"].nunique().max() == 1, "cluster_40 spans both folds"
assert mech_s2.groupby("homology_group")["lso_fold"].nunique().max() == 1, "homology group spans both folds"
fold_s2 = mech_s2.set_index("sequence")["lso_fold"]
assert all(fold_s2[x] == fold_s2[y] for x, y in near_duplicate_pairs(fold_s2.index)), "near-duplicate pair spans the folds in S2"
# mung-bean duals must be exactly the ones the per-peptide tables support
mung_in_s2 = s2[s2["sequence"].isin(MUNG_EVIDENCE) & ~s2["sequence"].isin(EXCLUDED_NO_ACTIVITY)]
assert (mung_in_s2["labels"] == "FRS;MC").all(), "every retained mung-bean peptide must be dual (validated)"
print("All S2 invariants passed.")

# ---- backup + change log, THEN write ---------------------------------------------------------------
s2_path = f"{PROC}/AOP-BenchPos_S2.csv"
bak = f"{PROC}/AOP-BenchPos_S2.previous.csv"
if os.path.exists(s2_path) and not os.path.exists(bak):
    import shutil; shutil.copy(s2_path, bak); print("backed up previous S2 ->", bak)
prev_path = PREVIOUS_S2_PATH if PREVIOUS_S2_PATH else (bak if os.path.exists(bak) else None)
if prev_path and os.path.exists(prev_path):
    old = pd.read_csv(prev_path)
    m_ = old.merge(s2, on="sequence", how="outer", suffixes=("_old", "_new"), indicator=True)
    both = m_["_merge"] == "both"
    lab_ch = both & (m_["labels_old"] != m_["labels_new"])
    fold_ch = both & ~lab_ch & (m_["lso_fold_old"].fillna(-1) != m_["lso_fold_new"].fillna(-1))
    kind = np.select([m_["_merge"] == "left_only", m_["_merge"] == "right_only", lab_ch, fold_ch],
                     ["removed", "added", "label_changed", "fold_changed_only"], default="unchanged")
    log = m_.assign(change=kind)[kind != "unchanged"][["sequence", "change", "labels_old", "labels_new", "lso_fold_old", "lso_fold_new"]]
    log.to_csv(f"{PROC}/s2_change_log.csv", index=False)
    print("S2 versus", prev_path, "->", log["change"].value_counts().to_dict())
    print("  removed sequences by cause: Tier-3 conflicts =", int(log.loc[log["change"] == "removed", "sequence"].isin(CONFLICT_SET).sum()),
          "| others =", int((~log.loc[log["change"] == "removed", "sequence"].isin(CONFLICT_SET)).sum()))
else:
    print("No previous S2 found -> no change log. Set PREVIOUS_S2_PATH (Section 1.0 constants) to diff against an earlier S2 file.")
s2.to_csv(s2_path, index=False)
print("Saved:", s2_path, s2.shape)

In [ ]:
# What the labels actually rest on -- this is the table to show a supervisor
mcs = aop_benchpos_s2[aop_benchpos_s2["labels"].str.contains("MC")]
print("MC-active:", len(mcs))
print(pd.crosstab(mcs["source_group"], mcs["mc_evidence"], margins=True).to_string())
print("\nclass x source_group (mechanism-active):")
print(pd.crosstab(mech_s2["labels"], mech_s2["source_group"], margins=True).to_string())
print("\nlength (median) by class and source_group -- a visible confound to control for downstream:")
print(mech_s2.groupby(["labels", "source_group"])["length"].median().unstack().round(1).to_string())
strict = mech_s2[mech_s2["strict_evidence"] == True]
print(f"\nSTRICT-evidence subset (primary analysis): {len(strict)} sequences | MC-active {int(strict['labels'].str.contains('MC').sum())} "
      f"| FRS-active {int(strict['labels'].str.contains('FRS').sum())}")
print(f"EXTENDED set adds {int((mech_s2['strict_evidence'] == False).sum())} candidate-evidence sequences (sensitivity analysis only): {len(mech_s2)} in total")
res = mech_s2[mech_s2["homology_resolvable"]]
print(f"homology-resolvable (>= {NEAR_MIN_LEN} aa): {len(res)} of {len(mech_s2)} | MC-active among them: {int(res['labels'].str.contains('MC').sum())}")
n_dual_val = int((aop_benchpos_s2["labels"] == "FRS;MC").sum())
print(f"\nDual (FRS;MC) sequences: {n_dual_val}  (AnOxPePred-reported + validated mung-bean)")

---
## 1.7b Confounds that every downstream result must be read against (STEP 08 findings N-19, N-21)

This cell **changes no data and no file** (the S2 fingerprint is untouched); it only prints facts about S2 that decide how the probes and retrieval results of Steps 5-7 may be interpreted:

* **Source.** Every FRS-only peptide comes from AnOxPePred; a large part of the strict MC-active peptides comes from external literature. The single binary feature `source_group == external_literature` therefore separates MC-active from FRS-only peptides about as well as an ESM-2 probe (exact AUROC printed below).
* **Length.** The class means are almost equal; the difference is the **share of tripeptides** (and tripeptides cannot be homology-grouped, see V16). Earlier notes that said "MC peptides are longer (9.4 vs 5.8 aa)" were wrong for the strict pool.
* Consequence: report SOURCE-only and LENGTH-only baselines next to every probe, use the AnOxPePred-only / length-matched checks of STEP 08, and do not read the probe score as mechanism evidence.


In [ ]:
# read-only diagnostics on the S2 table built above (no file is written, no column is changed)
_s = aop_benchpos_s2[aop_benchpos_s2["strict_evidence"].eq(True)].copy()
_s["y"] = _s["labels"].str.contains("MC").astype(int)
_s["ext"] = (_s["source_group"] == "external_literature").astype(int)
print(f"strict pool: n={len(_s)}  MC-active={int(_s.y.sum())}  FRS-only={int((1 - _s.y).sum())}")
print()
print("label x source_group:")
print(pd.crosstab(_s["labels"], _s["source_group"], margins=True).to_string())
print()
print("length: mean (median)  FRS-only {:.2f} ({:.0f})   MC-active {:.2f} ({:.0f})".format(
      _s.loc[_s.y == 0, "length"].mean(), _s.loc[_s.y == 0, "length"].median(), _s.loc[_s.y == 1, "length"].mean(), _s.loc[_s.y == 1, "length"].median()))
print("tripeptide share:  FRS-only {:.1%}   MC-active {:.1%}".format((_s.loc[_s.y == 0, "length"] == 3).mean(), (_s.loc[_s.y == 1, "length"] == 3).mean()))
_p1, _p0 = _s.loc[_s.y == 1, "ext"].mean(), _s.loc[_s.y == 0, "ext"].mean()      # AUROC of one binary feature, no fitting
_auc = _p1 * (1 - _p0) + 0.5 * (_p1 * _p0 + (1 - _p1) * (1 - _p0))
print()
print(f"exact AUROC of the single feature source_group (external literature vs AnOxPePred) for MC-active vs FRS-only: {_auc:.3f}")
assert _s.loc[_s.y == 0, "ext"].sum() == 0, "FRS-only peptides are expected to be 100 % AnOxPePred in this build"


---
## 1.8 Negatives — `AOP-Inactive.csv` (V8)
AnOxPePred's authors report **218 "experimentally-validated non-antioxidant" peptides** (FRS = 0 and Chelator = 0). They were discarded in earlier versions. They are
the only *real* negatives in the project, so they are kept as a separate file for evaluation (e.g. "mechanism-active vs confirmed-inactive"), **not** mixed into S2.

**Use them with care (V20):** most of these peptides are *also* listed as "antioxidant" by DFBP / Multi-AOP (§1.5b). The **clean** negatives are those with `db_conflict == False`; use them for the primary
evaluation and report the full set as a sensitivity analysis.

They get the same leak-safe treatment so they can be used alongside S2 without leaking:
* a sequence identical to anything that is **still** in Tier 1/2/3 is dropped (a contradiction in the sources). Tier-3 sequences removed in §1.5b are therefore *kept* here, but flagged `db_conflict = True`;
* an inactive peptide that is a **near-duplicate of an active** inherits that active's fold; if its near-duplicates sit in *both* folds it is dropped as ambiguous;
* inactive peptides linked only to each other are grouped and balanced across folds.

In [ ]:
act_fold = (aop_benchpos[tier12_mask].drop_duplicates("sequence").set_index("sequence")["lso_fold"]).to_dict()
all_known = set(aop_benchpos["sequence"])

ina = anox_inactive.copy()
ina = ina[(ina["sequence"].str.len() >= 3) & (ina["sequence"].str.len() <= 50)]
n_in = len(ina)
clash = ina["sequence"].isin(all_known)
print(f"inactive controls after length filter: {n_in} | identical to a Tier1/2/3 sequence (dropped): {int(clash.sum())}")
ina = ina[~clash].drop_duplicates("sequence").reset_index(drop=True)

# link inactives to actives and to each other with the same near-duplicate rule
ina_seqs = ina["sequence"].tolist()
uf = UnionFind()
for s in ina_seqs: uf.find(s)
anchor = {s: set() for s in ina_seqs}                                   # folds of near-duplicate actives
act_seqs = list(act_fold)
for s in ina_seqs:
    for t in act_seqs:
        if is_near_duplicate(s, t): anchor[s].add(int(act_fold[t]))
for a_, b_ in near_duplicate_pairs(ina_seqs): uf.union(a_, b_)

comp = {}
for s in ina_seqs: comp.setdefault(uf.find(s), []).append(s)
rows, dropped_ambiguous, free_comps = [], 0, []
for root, members in sorted(comp.items(), key=lambda kv: kv[0]):
    folds = set().union(*[anchor[m] for m in members])
    if len(folds) > 1:
        dropped_ambiguous += len(members); continue
    if len(folds) == 1:
        f = next(iter(folds)); rows += [(m, root, f) for m in members]
    else:
        free_comps.append((root, members))
load = [sum(1 for r in rows if r[2] == k) for k in (0, 1)]
for root, members in sorted(free_comps, key=lambda c: (-len(c[1]), c[0])):
    f = int(np.argmin(load)); rows += [(m, root, f) for m in members]; load[f] += len(members)

inactive_df = pd.DataFrame(rows, columns=["sequence", "_root", "lso_fold"])
inactive_df["homology_group"] = inactive_df["_root"].astype("category").cat.codes
inactive_df = inactive_df.drop(columns="_root")
inactive_df["db_conflict"] = inactive_df["sequence"].isin(DB_LISTED_SET)     # V20: listed "antioxidant" by a database ANYWHERE in the raw pool (not only the retained Tier 3)
inactive_df["source_db"] = "AnOxPePred_nonantioxidant"
inactive_df["evidence"] = "experimentally_validated_non_antioxidant (Olsen et al. 2020)"
inactive_df["length"] = inactive_df["sequence"].str.len()
inactive_df = inactive_df.sort_values("sequence").reset_index(drop=True)

assert inactive_df["sequence"].is_unique and not (set(inactive_df["sequence"]) & all_known)
assert inactive_df.groupby("homology_group")["lso_fold"].nunique().max() == 1
print(f"dropped as ambiguous (near-duplicates in both folds): {dropped_ambiguous}")
print("AOP-Inactive:", inactive_df.shape, "| per fold:", inactive_df["lso_fold"].value_counts().sort_index().to_dict())
print(f"  db_conflict = True (a database lists it as antioxidant, raw pool): {int(inactive_df['db_conflict'].sum())} | "
      f"CLEAN negatives (no database disagrees): {int((~inactive_df['db_conflict']).sum())}")
inactive_df.to_csv(f"{PROC}/AOP-Inactive.csv", index=False)
print("Saved: AOP-Inactive.csv")

---
## 1.9 Manifest, fingerprint and what must be re-run
The manifest records counts, the SHA-256 of every output, the S2 fingerprint and the git commits of the cloned sources, so a later run can be compared
with this one. `EXPECTED_FINGERPRINT` is left `None` on a first run: paste the printed value into it after you have inspected the outputs, and every later
run will then report MATCH / DIFFERENT. (A *DIFFERENT* result is not automatically an error — live sources can change — but it must be explained.)

In [ ]:
EXPECTED_FINGERPRINT = "47d9c2a2fff63eb1"   # derived by replaying this notebook on the archived Drive data; a live run should match -- if not, explain why

def s2_fingerprint(s2_df):
    """Canonical content hash of S2. It covers sequence, labels, homology_group, lso_fold and sample_weight, and is independent of
    row order, dtypes, how missing values are spelled, and of CD-HIT's arbitrary cluster numbering (cluster_40 ids depend on the
    order of the FASTA input, so they are deliberately NOT hashed -- the homology GROUPS they feed are)."""
    lines = []
    for r in s2_df.sort_values("sequence").itertuples():
        lines.append("\t".join([r.sequence, r.labels,
                                "" if pd.isna(r.homology_group) else str(int(r.homology_group)),
                                "" if pd.isna(r.lso_fold) else str(int(r.lso_fold)),
                                f"{float(r.sample_weight):.1f}"]))
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16]

fingerprint = s2_fingerprint(aop_benchpos_s2)

def sha(path): return hashlib.sha256(open(path, "rb").read()).hexdigest()
outputs = ["AOP-BenchPos.csv", "AOP-BenchPos_S2.csv", "AOP-Inactive.csv", "AOP-Provenance.csv", "AOP-Conflicts.csv", "excluded_by_primary_evidence.csv", "merged_raw.csv"]
manifest = {
    "provenance": PROVENANCE,
    "counts": {
        "S1_rows": int(len(aop_benchpos)), "S2_rows": int(len(aop_benchpos_s2)),
        "labels": aop_benchpos_s2["labels"].value_counts().to_dict(),
        "tier2_total": int(len(tier2_final)),
        "mc_evidence": aop_benchpos_s2.loc[aop_benchpos_s2["labels"].str.contains("MC"), "mc_evidence"].value_counts().to_dict(),
        "inactive_rows": int(len(inactive_df)),
        "excluded_by_primary_evidence": sorted(EXCLUDED_NO_ACTIVITY),
        "near_duplicate_pairs_linked": int(len(near_pairs)),
        "conflicts_with_anox_negatives": {"policy": CONFLICT_POLICY, "retained_tier3_sequences": int(len(CONFLICT_SET)),
                                          "raw_pool_sequences": int(len(DB_LISTED_SET))},
        "clean_inactive_negatives": int((~inactive_df["db_conflict"]).sum()),
        "homology_groups": int(len(set(seq_group.values()))),
        "largest_homology_group": int(pd.Series(list(seq_group.values())).value_counts().iloc[0]),
    },
    "split_type": "homology-grouped, MC-stratified greedy 2-fold split (NOT leave-source-out; column name lso_fold kept for backward compatibility)",
    "near_duplicate_rule": f"identity >= {NEAR_IDENT:.0%} (1 - Levenshtein / longer length), at least one sequence >= {NEAR_MIN_LEN} aa (our own normalised-edit rule, not AnOxPePred's alignment identity); plus CD-HIT 40% (shorter-sequence denominator) for >= 10 aa",
    "split_audit": AUDIT,
    "strict_evidence_counts": {
        "strict_sequences": int((aop_benchpos_s2["strict_evidence"] == True).sum()),
        "candidate_sequences": int((aop_benchpos_s2["strict_evidence"] == False).sum()),
        "strict_MC_active": int(((aop_benchpos_s2["strict_evidence"] == True) & aop_benchpos_s2["labels"].str.contains("MC")).sum()),
    },
    "s2_fingerprint": fingerprint,
    "sha256": {f: sha(f"{PROC}/{f}") for f in outputs if os.path.exists(f"{PROC}/{f}")},
}
json.dump(manifest, open(f"{PROC}/step1_manifest.json", "w"), indent=2, default=str)
print(json.dumps(manifest["counts"], indent=2, default=str))
print("\nS2 fingerprint:", fingerprint)
if EXPECTED_FINGERPRINT is None:
    print("-> first run: inspect the outputs, then paste this fingerprint into EXPECTED_FINGERPRINT above.")
else:
    print("-> matches the recorded run." if fingerprint == EXPECTED_FINGERPRINT else
          "-> DIFFERENT from the recorded run: a live source or CD-HIT output changed -- explain before proceeding.")

In [ ]:
n_old = None
bak = f"{PROC}/AOP-BenchPos_S2.previous.csv"
if os.path.exists(bak): n_old = len(pd.read_csv(bak))
print("S2 rows now:", len(aop_benchpos_s2), "| previous S2 rows:", n_old)
print("""
DOWNSTREAM -- what this run invalidates
  Step 2  descriptors.csv / esm2_*.npy : row sets changed -> re-align by SEQUENCE (embeddings depend on the sequence only, not on labels or folds).
                                          s2_change_log.csv lists exactly which sequences were removed/added.
  Step 3  pair files                    : the MC / FRS pools changed -> re-run with OVERWRITE = True (also redraws ablation_random).
  Step 4  models                        : folds changed -> retrain all models.
  Step 5  evaluation / probe            : use `homology_group` (not cluster_40) for any grouped CV; use `source_group` and `length` as controls;
                                          AOP-Inactive.csv supplies real negatives.
  NOTE   Steps 3-5 as delivered do NOT yet filter on `strict_evidence`, `homology_resolvable` or `db_conflict` -- until they are patched, none of the
         'primary analysis' recommendations above is actually enforced downstream.
""")